# A Beginner's Guide to Causal Inference with DoWhy in Python

This notebook is a runnable companion to the blog post [*A Beginner's Guide to Causal Inference with DoWhy in Python*](https://carlos-mendez.org/post/python_dowhy_intro/).

Does working from home actually make employees more productive, or do more productive people simply *choose* to work from home? This is the fundamental challenge of **causal inference**: distinguishing genuine cause-and-effect relationships from misleading correlations driven by confounding variables.

In this notebook, we use **simulated observational data** where the true causal effect is known (ATE = 1.0) to demonstrate how **[DoWhy](https://www.pywhy.org/dowhy/v0.14/)** --- a Python library for causal inference --- helps us recover the correct answer. We walk through DoWhy's four-step framework (**Model, Identify, Estimate, Refute**) and apply four estimation methods.

> **How to use this notebook:** click `Runtime → Run all` (or press `Ctrl+F9`) to execute every cell. The full pipeline takes under 1 minute on a free Colab CPU runtime.

**Learning objectives:**

- Understand why naive comparisons can be misleading when confounders are present
- Learn DoWhy's four-step causal inference framework (Model, Identify, Estimate, Refute)
- Define a causal graph (DAG) that encodes your assumptions about what causes what
- Distinguish two identification strategies: **selection on observables** (backdoor criterion) and **instrumental variables**
- Estimate causal effects using four methods and compare them against the known truth
- Assess robustness of estimates using automated refutation tests

## Setup and imports

Install the required packages:

In [ ]:
!pip install dowhy statsmodels --quiet

Import all necessary libraries and set configuration variables.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression, LinearRegression
from dowhy import CausalModel
import statsmodels.api as sm

# Configuration
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

N = 5000           # sample size
TRUE_ATE = 1.0     # known true effect

TREATMENT = "work_from_home"
OUTCOME = "productivity"
CONFOUNDERS = ["introversion", "num_children"]
INSTRUMENT = "company_policy"

# Site color palette
STEEL_BLUE = "#6a9bcc"
WARM_ORANGE = "#d97757"
NEAR_BLACK = "#141413"
TEAL = "#00d4c8"
PURPLE = "#8b5cf6"

## The data: simulated observational study

We simulate data where the **true causal effect is known** (ATE = 1.0), so we can verify whether each method recovers the correct answer. This is the gold standard for learning causal inference: if a method cannot recover the truth when we know it, we should not trust it on real data where we do not.

### Data generating process

The data generating process (DGP) defines how each variable is generated. Think of it as the "rules of the universe" in our simulation. Notice three critical features:

1. **Treatment is NOT random.** More introverted employees and those with more children are more likely to choose WFH.
2. **Confounders affect both treatment and outcome.** Introversion increases both the probability of WFH and productivity directly.
3. **The instrument satisfies the exclusion restriction.** `company_policy` appears in the treatment equation but NOT in the outcome equation.

In [ ]:
def generate_wfh_data(n, seed):
    """Simulate observational data on working from home and productivity."""
    rng = np.random.default_rng(seed)

    # Confounders (affect BOTH treatment and outcome)
    introversion = rng.normal(5, 1.5, n)
    num_children = rng.poisson(1.5, n)

    # Instrument (affects treatment ONLY)
    company_policy = rng.binomial(1, 0.4, n)

    # Treatment: who works from home? (observational, not random!)
    logit_p = -1.5 + 0.3*introversion + 0.2*num_children + 1.0*company_policy
    prob_wfh = 1 / (1 + np.exp(-logit_p))
    work_from_home = rng.binomial(1, prob_wfh)

    # Outcome: productivity (note: company_policy does NOT appear here)
    noise = rng.normal(0, 2, n)
    productivity = (50
                    + 1.0 * work_from_home      # TRUE causal effect
                    + 0.8 * introversion         # confounder effect
                    - 0.5 * num_children         # confounder effect
                    + noise)

    return pd.DataFrame({
        "work_from_home": work_from_home,
        "productivity": productivity,
        "introversion": introversion,
        "num_children": num_children,
        "company_policy": company_policy,
    })

df = generate_wfh_data(N, RANDOM_SEED)
print(f"Dataset shape: {df.shape}")
print(f"Treatment prevalence: {df[TREATMENT].mean():.1%} work from home")
print(f"\nDescriptive statistics:")
df.describe().round(2)

Alternatively, you can load the pre-generated dataset directly from GitHub:

In [ ]:
# Uncomment the line below to load from GitHub instead of generating locally
# df = pd.read_csv("https://raw.githubusercontent.com/carlos-mendez/starter-academic-v501/master/content/post/python_dowhy_intro/wfh_simulated_data.csv")
# df.head()

## Exploratory data analysis

### The naive estimate

The simplest approach is to compare average productivity between the two groups:

In [ ]:
mean_wfh = df[df[TREATMENT] == 1][OUTCOME].mean()
mean_office = df[df[TREATMENT] == 0][OUTCOME].mean()
naive_ate = mean_wfh - mean_office

print(f"Mean productivity (WFH):    {mean_wfh:.2f}")
print(f"Mean productivity (Office): {mean_office:.2f}")
print(f"Naive ATE (difference):     {naive_ate:.2f}")
print(f"True ATE:                   {TRUE_ATE:.2f}")
print(f"Bias (naive - true):        {naive_ate - TRUE_ATE:.2f}")

The naive estimate overshoots the true effect of 1.0 by about 39%. This upward bias occurs because the WFH group contains more introverts, who are independently more productive.

### Visualizing the confounding

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Panel A: Productivity distribution by treatment
ax = axes[0]
for group, label, color in [(0, "Office", STEEL_BLUE), (1, "WFH", WARM_ORANGE)]:
    subset = df[df[TREATMENT] == group][OUTCOME]
    ax.hist(subset, bins=35, alpha=0.6, label=f"{label} (mean={subset.mean():.1f})",
            color=color, edgecolor="white")
    ax.axvline(subset.mean(), color=color, linewidth=2, linestyle="--")
ax.axvline(mean_office + TRUE_ATE, color=NEAR_BLACK, linewidth=1.5, linestyle=":",
           label=f"True causal effect (Office mean + {TRUE_ATE})")
ax.set_xlabel("Productivity")
ax.set_ylabel("Count")
ax.set_title("A. Productivity Distribution by Group")
ax.legend(fontsize=9)

# Panel B: Covariate imbalance
ax = axes[1]
x = np.arange(len(CONFOUNDERS))
width = 0.35
means_office = df[df[TREATMENT] == 0][CONFOUNDERS].mean()
means_wfh = df[df[TREATMENT] == 1][CONFOUNDERS].mean()
ax.bar(x - width/2, means_office, width, label="Office", color=STEEL_BLUE, edgecolor="white")
ax.bar(x + width/2, means_wfh, width, label="WFH", color=WARM_ORANGE, edgecolor="white")
ax.set_xticks(x)
ax.set_xticklabels(["Introversion", "Num. Children"])
ax.set_ylabel("Mean Value")
ax.set_title("B. Covariate Imbalance (Confounders)")
ax.legend()
ax.annotate("Self-selection\nbias!", xy=(0, means_wfh.iloc[0]),
            xytext=(0.3, means_wfh.iloc[0] + 0.3),
            fontsize=9, color=WARM_ORANGE, fontweight="bold",
            arrowprops=dict(arrowstyle="->", color=WARM_ORANGE))

plt.suptitle("Observational Data: Confounders Create Selection Bias",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

**Panel A** shows the productivity distributions for office (blue) and WFH (orange) workers. The WFH distribution is shifted right, but not all of this shift is causal --- part of it reflects the higher introversion of WFH workers.

**Panel B** reveals the covariate imbalance: WFH employees have higher introversion and more children than office workers. This imbalance is the fingerprint of self-selection and the source of confounding bias.

## Step 1: Model --- Define the causal graph

The first step in DoWhy is to encode your **causal assumptions** as a Directed Acyclic Graph (DAG). Three types of variables appear in our graph:

- **Confounders** (gray): Introversion and num_children have arrows to *both* treatment and outcome
- **Instrument** (teal): Company policy has an arrow to treatment but *not* to outcome
- **Treatment and Outcome** (orange and blue): The arrow from treatment to outcome is the causal effect we want to estimate

In [ ]:
model = CausalModel(
    data=df,
    treatment=TREATMENT,
    outcome=OUTCOME,
    common_causes=CONFOUNDERS,
    instruments=[INSTRUMENT],
)
print("CausalModel created successfully.")
print(f"  Treatment: {TREATMENT}")
print(f"  Outcome: {OUTCOME}")
print(f"  Confounders: {CONFOUNDERS}")
print(f"  Instrument: {INSTRUMENT}")

Let's visualize the DAG:

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.set_xlim(-0.5, 3.5)
ax.set_ylim(-0.5, 3.5)
ax.axis("off")

# Node positions
nodes = {
    "company_policy": (0.5, 2.0),
    "work_from_home": (1.5, 2.0),
    "productivity": (3.0, 2.0),
    "introversion": (1.5, 3.2),
    "num_children": (2.5, 0.8),
}

# Draw nodes
for name, (x, y) in nodes.items():
    color = (TEAL if name == INSTRUMENT else
             (WARM_ORANGE if name == TREATMENT else
              (STEEL_BLUE if name == OUTCOME else "#999999")))
    ax.add_patch(plt.Circle((x, y), 0.25, fc=color, ec=NEAR_BLACK, alpha=0.8, zorder=2))
    label = name.replace("_", "\n")
    ax.text(x, y, label, ha="center", va="center", fontsize=8,
            fontweight="bold", color="white", zorder=3)

# Draw edges (arrows)
edges = [
    ("company_policy", "work_from_home"),
    ("work_from_home", "productivity"),
    ("introversion", "work_from_home"),
    ("introversion", "productivity"),
    ("num_children", "work_from_home"),
    ("num_children", "productivity"),
]
for src, dst in edges:
    x1, y1 = nodes[src]
    x2, y2 = nodes[dst]
    dx, dy = x2 - x1, y2 - y1
    length = np.sqrt(dx**2 + dy**2)
    shrink = 0.28
    ax.annotate("", xy=(x2 - shrink*dx/length, y2 - shrink*dy/length),
                xytext=(x1 + shrink*dx/length, y1 + shrink*dy/length),
                arrowprops=dict(arrowstyle="->", lw=2, color=NEAR_BLACK))

ax.set_title("Causal DAG: Effect of Working from Home on Productivity",
             fontsize=12, fontweight="bold", pad=20)
plt.show()

## Step 2: Identify --- Find the estimand

**Identification** answers the question: *Can we express the causal effect as something we can compute from data?* This is purely a theoretical step --- it depends on the graph, not the data.

DoWhy automatically discovers two valid identification strategies:

1. **Backdoor criterion**: condition on all confounders to block backdoor paths
2. **Instrumental variable**: use `company_policy` to isolate exogenous variation in treatment

In [ ]:
identified_estimand = model.identify_effect(proceed_when_unidentifiable=True)
print(identified_estimand)

## Step 3: Estimate --- Compute the causal effect

Now we apply four estimation methods. Methods 1--3 use the backdoor criterion (selection on observables), while Method 4 uses the instrumental variable strategy.

### Method 1: Linear Regression (backdoor adjustment)

The simplest approach: include confounders as control variables in a regression.

$$Y_i = \beta_0 + \beta_1 T_i + \beta_2 X_{1i} + \beta_3 X_{2i} + \varepsilon_i$$

The coefficient $\beta_1$ is the causal effect, provided the model is correctly specified and all confounders are included.

In [ ]:
# DoWhy estimation
estimate_reg = model.estimate_effect(
    identified_estimand,
    method_name="backdoor.linear_regression",
    confidence_intervals=True,
)

# Robust SE via statsmodels OLS with HC1
X_reg = sm.add_constant(df[[TREATMENT] + CONFOUNDERS])
ols_model = sm.OLS(df[OUTCOME], X_reg).fit(cov_type="HC1")
reg_ate = ols_model.params[TREATMENT]
se_reg = ols_model.bse[TREATMENT]
ci_reg = (ols_model.conf_int().loc[TREATMENT, 0], ols_model.conf_int().loc[TREATMENT, 1])

print(f"Estimated ATE: {reg_ate:.4f}")
print(f"Robust SE (HC1): {se_reg:.4f}")
print(f"95% CI: [{ci_reg[0]:.4f}, {ci_reg[1]:.4f}]")
print(f"Bias from true ({TRUE_ATE}): {reg_ate - TRUE_ATE:.4f}")

Linear regression recovers the true effect almost exactly (bias < 1%). The robust standard error (HC1) accounts for potential heteroskedasticity.

### Method 2: Inverse Probability Weighting (IPW)

IPW models the **treatment assignment mechanism** (who gets treated and why). It weights each observation by the inverse of the probability of receiving its actual treatment, creating a pseudo-population where treatment is independent of confounders.

$$\widehat{ATE}_{IPW} = \frac{1}{N}\sum_{i=1}^{N}\left[\frac{T_i Y_i}{\hat{e}(X_i)} - \frac{(1 - T_i) Y_i}{1 - \hat{e}(X_i)}\right]$$

where $\hat{e}(X_i) = P(T_i = 1 \mid X_i)$ is the **propensity score**.

In [ ]:
# Fit propensity score model
ps_model = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)
ps_model.fit(df[CONFOUNDERS], df[TREATMENT])
ps = ps_model.predict_proba(df[CONFOUNDERS])[:, 1]

T = df[TREATMENT].values
Y = df[OUTCOME].values

# Hajek (normalized) IPW estimator
w1 = 1.0 / ps
w0 = 1.0 / (1 - ps)
mu1_ipw = np.sum(T * Y * w1) / np.sum(T * w1)
mu0_ipw = np.sum((1 - T) * Y * w0) / np.sum((1 - T) * w0)
ipw_ate = mu1_ipw - mu0_ipw

# Influence function for robust SE
ipw_phi_1 = T * w1 * (Y - mu1_ipw) / np.mean(T * w1)
ipw_phi_0 = (1 - T) * w0 * (Y - mu0_ipw) / np.mean((1 - T) * w0)
ipw_phi = ipw_phi_1 - ipw_phi_0
se_ipw = np.std(ipw_phi, ddof=1) / np.sqrt(N)
ci_ipw = (ipw_ate - 1.96 * se_ipw, ipw_ate + 1.96 * se_ipw)

print(f"Estimated ATE: {ipw_ate:.4f}")
print(f"Robust SE (influence function): {se_ipw:.4f}")
print(f"95% CI: [{ci_ipw[0]:.4f}, {ci_ipw[1]:.4f}]")
print(f"Bias from true ({TRUE_ATE}): {ipw_ate - TRUE_ATE:.4f}")

IPW recovers the true effect with a bias of only about 3%. Its SE is slightly larger than regression because IPW discards the outcome model entirely and relies solely on propensity scores.

### Method 3: Doubly Robust (AIPW)

The **doubly robust** estimator combines both outcome modeling AND propensity score weighting. It is consistent if *either* model is correctly specified.

$$\widehat{ATE}_{DR} = \frac{1}{N}\sum_{i=1}^{N}\left[(\hat{\mu}_1(X_i) - \hat{\mu}_0(X_i)) + \frac{T_i(Y_i - \hat{\mu}_1(X_i))}{\hat{e}(X_i)} - \frac{(1-T_i)(Y_i - \hat{\mu}_0(X_i))}{1 - \hat{e}(X_i)}\right]$$

In [ ]:
# Fit outcome models for each treatment group
outcome_model_1 = LinearRegression().fit(
    df[df[TREATMENT] == 1][CONFOUNDERS], df[df[TREATMENT] == 1][OUTCOME])
outcome_model_0 = LinearRegression().fit(
    df[df[TREATMENT] == 0][CONFOUNDERS], df[df[TREATMENT] == 0][OUTCOME])

# Predicted potential outcomes
mu1 = outcome_model_1.predict(df[CONFOUNDERS])
mu0 = outcome_model_0.predict(df[CONFOUNDERS])

# AIPW influence function
phi = (mu1 - mu0) + T * (Y - mu1) / ps - (1 - T) * (Y - mu0) / (1 - ps)
dr_ate = np.mean(phi)
se_dr = np.std(phi, ddof=1) / np.sqrt(N)
ci_dr = (dr_ate - 1.96 * se_dr, dr_ate + 1.96 * se_dr)

print(f"Estimated ATE: {dr_ate:.4f}")
print(f"Robust SE (influence function): {se_dr:.4f}")
print(f"95% CI: [{ci_dr[0]:.4f}, {ci_dr[1]:.4f}]")
print(f"Bias from true ({TRUE_ATE}): {dr_ate - TRUE_ATE:.4f}")

The doubly robust estimate sits between regression and IPW. When both models are correctly specified, DR achieves the semiparametric efficiency bound --- the smallest possible variance.

### Method 4: Instrumental Variables (2SLS)

Methods 1--3 all rely on **selection on observables**. But what if there are unmeasured confounders? **Instrumental Variables (IV)** offers a solution using a variable that:

1. Affects the treatment (relevance)
2. Does NOT directly affect the outcome (exclusion restriction)
3. Is not caused by confounders (independence)

$$ATE = \frac{\text{Reduced form (effect of Z on Y)}}{\text{First stage (effect of Z on T)}}$$

In [ ]:
Z = df[INSTRUMENT].values

# Reduced form: effect of instrument on outcome
X_rf = sm.add_constant(df[[INSTRUMENT]])
reduced_form = sm.OLS(df[OUTCOME], X_rf).fit(cov_type="HC1")
gamma = reduced_form.params[INSTRUMENT]
se_gamma = reduced_form.bse[INSTRUMENT]

# First stage: effect of instrument on treatment
first_stage = sm.OLS(df[TREATMENT], X_rf).fit(cov_type="HC1")
pi_coef = first_stage.params[INSTRUMENT]
se_pi = first_stage.bse[INSTRUMENT]

# Wald IV estimate
iv_ate = gamma / pi_coef

# Delta-method robust SE
se_iv = (1 / abs(pi_coef)) * np.sqrt(se_gamma**2 + (gamma / pi_coef)**2 * se_pi**2)
ci_iv = (iv_ate - 1.96 * se_iv, iv_ate + 1.96 * se_iv)

print(f"First-stage F-statistic: {first_stage.fvalue:.1f}")
print(f"First-stage coefficient on {INSTRUMENT}: {pi_coef:.4f} (robust SE: {se_pi:.4f})")
print(f"Reduced-form coefficient: {gamma:.4f} (robust SE: {se_gamma:.4f})")
print(f"\nEstimated ATE: {iv_ate:.4f}")
print(f"Robust SE (HC1, delta method): {se_iv:.4f}")
print(f"95% CI: [{ci_iv[0]:.4f}, {ci_iv[1]:.4f}]")
print(f"Bias from true ({TRUE_ATE}): {iv_ate - TRUE_ATE:.4f}")

The IV estimate is far noisier (SE ~5x larger than regression), but it has a crucial advantage: it remains valid even with **unmeasured confounders**, as long as the exclusion restriction holds. The wide CI is the price of that robustness.

### Comparing all estimates

In [ ]:
# Store all results
results = {
    "Linear Regression": reg_ate,
    "IPW": ipw_ate,
    "Doubly Robust": dr_ate,
    "IV (2SLS)": iv_ate,
}
results_ci = {
    "Naive": (naive_ate - 1.96 * 0.0716, naive_ate + 1.96 * 0.0716),
    "Linear Regression": (float(ci_reg[0]), float(ci_reg[1])),
    "IPW": ci_ipw,
    "Doubly Robust": ci_dr,
    "IV (2SLS)": ci_iv,
}
results_se = {
    "Naive": 0.0716,
    "Linear Regression": float(se_reg),
    "IPW": se_ipw,
    "Doubly Robust": se_dr,
    "IV (2SLS)": se_iv,
}

# Compute naive SE properly
mean_wfh_vals = df[df[TREATMENT] == 1][OUTCOME]
mean_off_vals = df[df[TREATMENT] == 0][OUTCOME]
se_naive = np.sqrt(mean_wfh_vals.var() / len(mean_wfh_vals)
                   + mean_off_vals.var() / len(mean_off_vals))
results_ci["Naive"] = (naive_ate - 1.96 * se_naive, naive_ate + 1.96 * se_naive)
results_se["Naive"] = se_naive

# Summary table
print(f"{'Method':<25} {'Estimate':>9} {'Rob. SE':>9} {'95% CI':>22} {'Bias':>8} {'Covers?':>8}")
print("-" * 85)
print(f"{'True ATE':<25} {TRUE_ATE:>9.4f}")
all_methods = ["Naive", "Linear Regression", "IPW", "Doubly Robust", "IV (2SLS)"]
all_estimates = [naive_ate, reg_ate, ipw_ate, dr_ate, iv_ate]
for method, est in zip(all_methods, all_estimates):
    lo, hi = results_ci[method]
    se_val = results_se[method]
    bias = est - TRUE_ATE
    covers = "YES" if lo <= TRUE_ATE <= hi else "NO"
    print(f"{method:<25} {est:>9.4f} {se_val:>9.4f} [{lo:>8.4f}, {hi:>8.4f}] {bias:>+8.4f} {covers:>8}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

methods_list = ["Naive\n(Diff. in Means)", "Linear\nRegression", "IPW",
                "Doubly\nRobust", "IV\n(2SLS)"]
estimates_list = [naive_ate, reg_ate, ipw_ate, dr_ate, iv_ate]
ci_keys_list = ["Naive", "Linear Regression", "IPW", "Doubly Robust", "IV (2SLS)"]
ci_lower = [results_ci[k][0] for k in ci_keys_list]
ci_upper = [results_ci[k][1] for k in ci_keys_list]
xerr_left = [est - lo for est, lo in zip(estimates_list, ci_lower)]
xerr_right = [hi - est for est, hi in zip(estimates_list, ci_upper)]
colors = ["#999999", STEEL_BLUE, WARM_ORANGE, TEAL, PURPLE]

y_pos = np.arange(len(methods_list))

for i, (method, est, lo, hi, color) in enumerate(
        zip(methods_list, estimates_list, xerr_left, xerr_right, colors)):
    ax.errorbar(est, i, xerr=[[lo], [hi]], fmt="o", markersize=10,
                color=color, ecolor=color, elinewidth=2.5, capsize=6,
                capthick=2.5, zorder=4)

ax.axvline(TRUE_ATE, color=NEAR_BLACK, linewidth=2, linestyle="--",
           label=f"True ATE = {TRUE_ATE}", zorder=3)

for i, (est, ci_k) in enumerate(zip(estimates_list, ci_keys_list)):
    lo, hi = results_ci[ci_k]
    label = f"{est:.3f}  [{lo:.3f}, {hi:.3f}]"
    ax.text(max(hi, est) + 0.04, i, label, va="center", ha="left",
            fontsize=9, color=NEAR_BLACK)

ax.set_yticks(y_pos)
ax.set_yticklabels(methods_list)
ax.set_xlabel("Estimated Average Treatment Effect (ATE)")
ax.set_title("Causal Effect Estimates with 95% Confidence Intervals\n"
             "Effect of Working from Home on Productivity",
             fontsize=12, fontweight="bold")
ax.legend(loc="upper right", fontsize=11)
ax.set_xlim(0, max(ci_upper) + 0.55)
ax.axvspan(0.95, 1.05, alpha=0.08, color=NEAR_BLACK, zorder=1)

plt.tight_layout()
plt.show()

All four causal methods recover the true ATE far better than the naive estimate. The backdoor methods (Regression, IPW, DR) are more precise, while IV is noisier but does not require observing all confounders. Crucially, the naive CI does not contain the true ATE --- it is *confidently* wrong.

## Step 4: Refute --- Test robustness

The final step in DoWhy is **refutation** --- automated tests that check whether the estimate is robust to various challenges.

### Placebo treatment test

Randomly permute the treatment variable. If our method works correctly, the estimated effect should collapse to zero.

In [ ]:
# Create a backdoor-only model for refutation
model_backdoor = CausalModel(
    data=df,
    treatment=TREATMENT,
    outcome=OUTCOME,
    common_causes=CONFOUNDERS,
)
estimand_backdoor = model_backdoor.identify_effect(proceed_when_unidentifiable=True)
estimate_reg_bd = model_backdoor.estimate_effect(
    estimand_backdoor,
    method_name="backdoor.linear_regression",
)

# Placebo treatment
refute_placebo = model_backdoor.refute_estimate(
    estimand_backdoor,
    estimate_reg_bd,
    method_name="placebo_treatment_refuter",
    placebo_type="permute",
    num_simulations=100,
)
print(refute_placebo)

The placebo effect is essentially zero, confirming that the original effect is not an artifact.

### Random common cause test

Add a randomly generated variable as an additional confounder. If the estimate is robust, it should not change.

In [ ]:
refute_random = model_backdoor.refute_estimate(
    estimand_backdoor,
    estimate_reg_bd,
    method_name="random_common_cause",
    num_simulations=100,
)
print(refute_random)

The estimate remains unchanged after adding a random confounder --- our estimate is not fragile.

### Data subset test

Re-estimate on a random 80% subsample of the data.

In [ ]:
refute_subset = model_backdoor.refute_estimate(
    estimand_backdoor,
    estimate_reg_bd,
    method_name="data_subset_refuter",
    subset_fraction=0.8,
    num_simulations=100,
)
print(refute_subset)

The subsample estimate is nearly identical to the full-sample estimate, confirming stability.

## Takeaways

1. **Naive comparisons are misleading** when confounders are present. The naive estimate was 39% too high.
2. **Selection on observables** (backdoor criterion) works when you have measured all confounders. Regression, IPW, and DR all recovered the true ATE within 3%.
3. **Instrumental variables** provide identification even with unmeasured confounders, at the cost of a 5x larger standard error.
4. **Doubly robust estimation** offers insurance against misspecification *and* near-optimal precision.
5. **Standard errors measure precision, not validity.** The naive estimate has a small SE but is biased.
6. **DoWhy's four-step framework** forces transparency: declare assumptions, verify identifiability, estimate, and test robustness.

## Exercises

1. **Break the exclusion restriction.** Modify the DGP so that `company_policy` directly affects `productivity` (e.g., add `+ 0.5 * company_policy` to the outcome equation). Re-run the IV estimate. Does it still recover the true ATE?

2. **Add an unmeasured confounder.** Add a new variable `self_discipline` that affects both `work_from_home` and `productivity`, but do NOT include it in `CONFOUNDERS`. Compare how the backdoor methods (now biased) and IV (still valid) perform.

3. **Try a nonlinear DGP.** Replace the linear productivity equation with a nonlinear one (e.g., add an interaction term `0.3 * introversion * work_from_home`). Does linear regression still work well? What about the doubly robust estimator?

## References

- Sharma, A., & Kiciman, E. (2020). DoWhy: An end-to-end library for causal inference. *arXiv preprint arXiv:2011.04216*.
- Pearl, J. (2009). *Causality: Models, Reasoning, and Inference* (2nd ed.). Cambridge University Press.
- Angrist, J. D., & Pischke, J. S. (2009). *Mostly Harmless Econometrics*. Princeton University Press.
- Rosenbaum, P. R., & Rubin, D. B. (1983). The central role of the propensity score in observational studies for causal effects. *Biometrika*, 70(1), 41--55.
- Robins, J. M., Rotnitzky, A., & Zhao, L. P. (1994). Estimation of regression coefficients when some regressors are not always observed. *JASA*, 89(427), 846--866.

Full blog post: [carlos-mendez.org/post/python_dowhy_intro/](https://carlos-mendez.org/post/python_dowhy_intro/)